# PU Learning for Fraud Detection: when some fraud is never found

**Project 3 of 3**: Credit risk → Model monitoring → Financial crime (PU learning)

## Problem
In fraud and AML data, a "1" means *confirmed* fraud. A "0" does **not** mean clean: some fraud is never detected, so it hides among the 0s.
A normal model treats these hidden frauds as good examples of normal behaviour, and learns to score fraud-like cases lower.

This is called **Positive-Unlabeled (PU) learning**: we have labeled positives, and unlabeled data that mixes negatives with hidden positives.

## Approach
1. Take a fraud dataset where all labels are known, and **hide half of the fraud labels** to simulate undetected fraud. Because we still know the truth, we can measure how much each method recovers.
2. **Naive model**: treat every unlabeled row as non-fraud.
3. **PU model** (Elkan & Noto, 2008), two steps:
   - Step 1: a first model scores every row: how much does it look like the labeled fraud?
   - Step 2: unlabeled rows that look like fraud get a **lower weight as negatives** (and part of their weight as positives). Then the final model is trained with these weights.

This is the method I used at KPMG Japan for financial statement fraud, where many fraud cases are never detected.

## Data
[Bank Account Fraud Dataset Suite (NeurIPS 2022)](https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022), `Base.csv`: 1M bank account applications over 8 months, about 1% fraud.

```bash
pip install lightgbm scikit-learn pandas matplotlib
kaggle datasets download -d sgpjesus/bank-account-fraud-dataset-neurips-2022 -p data/raw/bank_account_fraud/
```

## 1. Setup

In [ ]:
import os
import time
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

COLORS = {"oracle": "#8a8985", "naive": "#eb6834", "PU": "#2a78d6", "PU downweight": "#1baf7a"}
NEG_COLOR, POS_COLOR, HIDDEN_COLOR = "#a3a3a3", "#e07b72", "#6250d6"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True, "axes.titleweight": "bold",
})

### Config
- **Out-of-time split**: train on months 0–5, test on months 6–7.
- `LABEL_FREQ = 0.5`: half of the fraud cases keep their label, the other half become "unlabeled" (simulated undetected fraud). This is the true **c**, the share of fraud that is labeled.
- In real data c is unknown. `ASSUMED_FRAUD_RATE` is a business guess of the real fraud rate; c is derived from it.

In [ ]:
DATA_PATH = os.environ.get("BAF_DATA_PATH", "../data/raw/bank_account_fraud/Base.csv")

TRAIN_MONTHS = [0, 1, 2, 3, 4, 5]
TEST_MONTHS = [6, 7]
LABEL_FREQ = 0.5              # true c in the simulation: share of fraud that stays labeled
ASSUMED_FRAUD_RATE = None     # business guess of the real fraud rate (None = use the true rate, simulation only)
N_FOLDS = 5                   # cross-fitting folds for step 1
RANDOM_STATE = 42

LGB_PARAMS = dict(
    n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
)

def make_model():
    return lgb.LGBMClassifier(**LGB_PARAMS)

## 2. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"{len(df):,} rows x {df.shape[1]} cols   fraud rate: {df['fraud_bool'].mean():.2%}")

# constant in Base.csv, so no information
df = df.drop(columns=[c for c in ["device_fraud_count"] if c in df.columns])

# in this dataset, negative values mean "missing" for these columns
NEG_IS_MISSING = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
                  "session_length_in_minutes", "device_distinct_emails_8w", "intended_balcon_amount"]
for c in NEG_IS_MISSING:
    if c in df.columns:
        df.loc[df[c] < 0, c] = np.nan

CAT_FEATURES = [c for c in ["payment_type", "employment_status", "housing_status", "source", "device_os"]
                if c in df.columns]
for c in CAT_FEATURES:
    df[c] = df[c].astype("category")

FEATURES = [c for c in df.columns if c not in ["fraud_bool", "month"]]
print(len(FEATURES), "features,", len(CAT_FEATURES), "categorical")
df.groupby("month")["fraud_bool"].agg(n="size", fraud_rate="mean")

## 3. Out-of-time split

In [ ]:
train = df[df["month"].isin(TRAIN_MONTHS)].reset_index(drop=True)
test = df[df["month"].isin(TEST_MONTHS)].reset_index(drop=True)

X_tr, y_tr = train[FEATURES], train["fraud_bool"].to_numpy()
X_te, y_te = test[FEATURES], test["fraud_bool"].to_numpy()

pd.DataFrame({"rows": [len(train), len(test)], "fraud_rate": [y_tr.mean(), y_te.mean()]},
             index=[f"train months {TRAIN_MONTHS[0]}-{TRAIN_MONTHS[-1]}", f"test months {TEST_MONTHS[0]}-{TEST_MONTHS[-1]}"])

## 4. Simulate undetected fraud
Each fraud case keeps its label with probability `LABEL_FREQ`. The rest become unlabeled (`s = 0`).
- `y`: true label (known only because this is a simulation)
- `s`: what we would see in real life (1 = confirmed fraud, 0 = unlabeled)

Labels are hidden **at random**. This is the main assumption of the method ("selected completely at random").

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
s_tr = ((y_tr == 1) & (rng.uniform(size=len(y_tr)) < LABEL_FREQ)).astype(int)
s_te = ((y_te == 1) & (rng.uniform(size=len(y_te)) < LABEL_FREQ)).astype(int)

hidden_tr = (y_tr == 1) & (s_tr == 0)
print(f"train: {y_tr.sum():,} frauds, {s_tr.sum():,} labeled, {hidden_tr.sum():,} hidden among {(s_tr == 0).sum():,} unlabeled rows")
print(f"true c (share of fraud labeled): {s_tr[y_tr == 1].mean():.3f}")

## 5. Baselines
- **Oracle**: trained on the true labels. Not possible in real life; it shows the best we could hope for.
- **Naive**: trained on `s`, i.e. every unlabeled row counts as non-fraud.

In [ ]:
t0 = time.time()
oracle = make_model().fit(X_tr, y_tr)
naive = make_model().fit(X_tr, s_tr)
print(f"fit {time.time() - t0:.0f}s")

## 6. PU step 1: score every row with a first model
The first model learns **labeled vs unlabeled**: g(x) = P(labeled | x).

Cross-fitting: each row is scored by a model that did not see it during training. Otherwise the model would remember the labeled rows and give them too high scores.

In [ ]:
g = np.zeros(len(s_tr))
folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
t0 = time.time()
for tr_idx, va_idx in folds.split(X_tr, s_tr):
    m = make_model().fit(X_tr.iloc[tr_idx], s_tr[tr_idx])
    g[va_idx] = m.predict_proba(X_tr.iloc[va_idx])[:, 1]
print(f"step 1 (cross-fit, {N_FOLDS} folds): {time.time() - t0:.0f}s")

# how do the three groups score? (we can only split "unlabeled" because this is a simulation)
groups = {"labeled fraud": (s_tr == 1), "hidden fraud (unlabeled)": hidden_tr, "true non-fraud": (y_tr == 0)}
colors = [POS_COLOR, HIDDEN_COLOR, NEG_COLOR]
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.logspace(-5, 0, 50)
for (name, mask), col in zip(groups.items(), colors):
    ax.hist(np.clip(g[mask], 1e-5, 1), bins=bins, density=True, histtype="step", lw=2, color=col, label=name)
ax.set_xscale("log")
ax.set(title="Step 1 score g(x) by group", xlabel="g(x) = P(labeled | x), log scale", ylabel="density")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 7. How much fraud is labeled? (c)
The weights need **c**, the share of all fraud that is labeled. Three ways to get it:
1. **From the data** (Elkan & Noto): average g(x) over labeled fraud. Only accurate when fraud and non-fraud are well separated.
2. **From a business guess** of the real fraud rate: c = (labeled rate) / (real fraud rate).
3. **True value**: only known in this simulation.

In practice, the best way is a **random audit**: investigate a random sample of unlabeled cases and count how many are fraud.

In [ ]:
c_data = g[s_tr == 1].mean()
fraud_rate_guess = ASSUMED_FRAUD_RATE if ASSUMED_FRAUD_RATE is not None else y_tr.mean()
c_prior = s_tr.mean() / fraud_rate_guess
c_true = s_tr[y_tr == 1].mean()

print(f"c from the data (Elkan & Noto): {c_data:.3f}")
print(f"c from the fraud-rate guess:    {c_prior:.3f}   (guess = {fraud_rate_guess:.2%})")
print(f"true c (simulation only):       {c_true:.3f}")

C = c_prior   # value used below

## 8. PU step 2: weights and the final model
For each unlabeled row, the chance that it is hidden fraud:

w(x) = (1 − c) / c × g(x) / (1 − g(x)), clipped to [0, 1]

- **PU (Elkan & Noto, "duplicate")**: each unlabeled row is used twice, as fraud with weight w and as non-fraud with weight 1 − w.
- **PU downweight**: each unlabeled row is non-fraud with weight 1 − w. Simpler: risky-looking unlabeled rows count less as "normal".

In [ ]:
def pu_weights(g, c, eps=1e-6):
    g = np.clip(g, eps, 1 - eps)
    return np.clip((1 - c) / c * g / (1 - g), 0, 1)


def weighted_training_set(X, s, w, method="duplicate"):
    pos, unl = (s == 1), (s == 0)
    parts_X = [X[pos], X[unl]]
    parts_y = [np.ones(pos.sum()), np.zeros(unl.sum())]
    parts_w = [np.ones(pos.sum()), 1 - w[unl]]
    if method == "duplicate":
        parts_X.append(X[unl])
        parts_y.append(np.ones(unl.sum()))
        parts_w.append(w[unl])
    return pd.concat(parts_X, ignore_index=True), np.concatenate(parts_y), np.concatenate(parts_w)


w = np.where(s_tr == 0, pu_weights(g, C), 0.0)

# do hidden frauds get higher weights than real non-fraud?
print(f"mean weight  hidden fraud: {w[hidden_tr].mean():.3f}   true non-fraud: {w[y_tr == 0].mean():.4f}")

t0 = time.time()
X_dup, y_dup, sw_dup = weighted_training_set(X_tr, s_tr, w, method="duplicate")
pu = make_model().fit(X_dup, y_dup, sample_weight=sw_dup)

X_dw, y_dw, sw_dw = weighted_training_set(X_tr, s_tr, w, method="downweight")
pu_down = make_model().fit(X_dw, y_dw, sample_weight=sw_dw)
print(f"step 2: {time.time() - t0:.0f}s")

## 9. Results on the test months (true labels)
- **AUC / PR-AUC (true)**: ranking against all real fraud, including hidden fraud.
- **Hidden recall @ top k**: among unlabeled test rows ranked by score, the share of hidden fraud found in the top k. This is the business question: "if analysts review the top 1% of unlabeled applications, how much of the undetected fraud do they find?"
- **Mean score**: average predicted probability. A well-scaled model should be close to the true fraud rate.

In [ ]:
def hidden_recall_at_k(y_true, s, score, k):
    unl = (s == 0)
    hidden = (y_true[unl] == 1)
    n_top = max(1, int(round(unl.sum() * k)))
    top = np.argsort(-score[unl])[:n_top]
    return hidden[top].sum() / hidden.sum()


scores = {
    "oracle": oracle.predict_proba(X_te)[:, 1],
    "naive": naive.predict_proba(X_te)[:, 1],
    "PU": pu.predict_proba(X_te)[:, 1],
    "PU downweight": pu_down.predict_proba(X_te)[:, 1],
}
results = pd.DataFrame({
    name: {
        "AUC (true)": roc_auc_score(y_te, p),
        "PR-AUC (true)": average_precision_score(y_te, p),
        "hidden recall @ top 1%": hidden_recall_at_k(y_te, s_te, p, 0.01),
        "hidden recall @ top 5%": hidden_recall_at_k(y_te, s_te, p, 0.05),
        "mean score": p.mean(),
    }
    for name, p in scores.items()
}).T
print(f"true fraud rate in test: {y_te.mean():.4f}")
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))

ax = axes[0]
for name, p in scores.items():
    prec, rec, _ = precision_recall_curve(y_te, p)
    ax.plot(rec * 100, prec * 100, color=COLORS[name], lw=2, ls="--" if name == "oracle" else "-",
            label=f"{name} (PR-AUC {average_precision_score(y_te, p):.3f})")
ax.axhline(y_te.mean() * 100, color=NEG_COLOR, ls=":", lw=1)
ax.set(title="Precision-recall on true labels (test)", xlabel="Recall (%)", ylabel="Precision (%)", xlim=(0, 100))
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
names = list(scores)
ax.bar(names, [scores[n].mean() * 100 for n in names], color=[COLORS[n] for n in names])
ax.axhline(y_te.mean() * 100, color=POS_COLOR, ls="--", lw=1.5, label=f"true fraud rate {y_te.mean():.2%}")
ax.set(title="Mean predicted probability vs true fraud rate", ylabel="%")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 10. How much does the assumed c matter?
In real life c is a guess. Step 1 does not depend on c, so we reuse g(x) and only retrain step 2 for each c.
If the results barely change, a rough business estimate of c is good enough.

In [ ]:
C_VALUES = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
rows = []
t0 = time.time()
for c in C_VALUES:
    w_c = np.where(s_tr == 0, pu_weights(g, c), 0.0)
    Xc, yc, swc = weighted_training_set(X_tr, s_tr, w_c, method="duplicate")
    p = make_model().fit(Xc, yc, sample_weight=swc).predict_proba(X_te)[:, 1]
    rows.append({"c": c, "PR-AUC (true)": average_precision_score(y_te, p),
                 "hidden recall @ top 5%": hidden_recall_at_k(y_te, s_te, p, 0.05),
                 "mean score": p.mean()})
sens = pd.DataFrame(rows).set_index("c")
print(f"{len(C_VALUES)} fits: {time.time() - t0:.0f}s")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for col, ax in zip(sens.columns, axes):
    ax.plot(sens.index, sens[col], marker="o", color=COLORS["PU"], label="PU")
    ax.axhline(results.loc["naive", col], color=COLORS["naive"], ls="--", lw=1.5, label="naive")
    if col == "mean score":
        ax.axhline(y_te.mean(), color=POS_COLOR, ls=":", lw=1.5, label="true fraud rate")
    ax.axvline(c_true, color="black", lw=1)
    ax.text(c_true, ax.get_ylim()[1], " true c", va="top", fontsize=8)
    ax.set(title=col, xlabel="assumed c")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()
sens

## 11. Business view: analysts review a fixed number of alerts
Each month analysts can review the top 1% of new applications. How many real frauds does each model put in front of them?

In [ ]:
k = 0.01
rows = []
for m in TEST_MONTHS:
    mask = (test["month"] == m).to_numpy()
    n_review = int(round(mask.sum() * k))
    row = {"month": m, "applications": int(mask.sum()), "reviewed": n_review}
    for name in ["naive", "PU"]:
        top = np.argsort(-scores[name][mask])[:n_review]
        row[f"frauds found ({name})"] = int(y_te[mask][top].sum())
    rows.append(row)
pd.DataFrame(rows).set_index("month")

## 12. Notes
Write your own observations here after running:
- How much did PU change PR-AUC and hidden recall compared with the naive model?
- Did the mean score move closer to the true fraud rate?
- How sensitive are the results to the assumed c?
- Limitations: labels were hidden at random. In real life, some fraud types are detected more often than others, which breaks this assumption.